# 1단계 태그 실험: 태그 입력 · 태그 엣지

`stage1_graph_experiments.ipynb`의 복사본입니다. 학습 절차·손실·5-폴드 교대·조기 종료·평가 지표는 원본과 같고, 현재 1단계 최적 설정(`weighted_mean`, 레이어 2, lr 0.003, hidden 64)에서 **태그 사용 방식만** 바꿉니다.

**사용하는 태그:** 목표가 "재료는 달라도 조리법·완성 형태가 비슷한 음식"이므로 재료 태그(`pork`, `chicken`, `vegetables` 등), 영양·식단 태그(`low-carb`, `vegetarian` 등), 모든 레시피에 붙는 상위 분류(`preparation`, `course` 등)는 뺍니다. 남는 것은 완성 형태(`main-dish`, `cakes`, `soups-stews` …), 조리법·도구(`oven`, `stove-top`, `crock-pot-slow-cooker` …), 시간·난이도, 국적, 행사 태그입니다.

| 조건 | 입력 특징 | 그래프 |
|---|---|---|
| `baseline` | 기존 `Recipe: {이름}. Steps: {조리 단계}` | 학습 라벨 엣지 |
| `tag_input` | `Recipe: {이름}. Tags: {태그}. Steps: {조리 단계}`를 GTE로 다시 인코딩 | 학습 라벨 엣지 |
| `tag_edges_k5` / `k10` | 기존 | 라벨 엣지 + 태그 IDF 코사인 상위 k개 이웃(양방향, 가중치 0.5 × 코사인) |
| `tag_input+tag_edges_k5` | 태그 포함 | 라벨 엣지 + 태그 엣지 k=5 |

- 태그 엣지는 라벨을 쓰지 않으므로 학습 폴드·검증·테스트 그래프에 모두 넣어도 정답이 새지 않습니다.
- 태그 입력은 `src.engine.GTETextEncoder`와 `training.stage1.recipe_text`를 쓰므로 **저장소 루트에서** 실행합니다(`data/recipes_raw_v3.csv`, `data/labeled_pairs_masked_v3.csv` 필요). 인코딩 결과는 `tmp/stage1_raw_gte_tags.pt`에 저장하고, 있으면 다시 만들지 않습니다.
- 한 실험 안의 비교는 같은 장치에서 합니다. 결과는 `results/experiments/04_tags/`에 저장합니다.


In [1]:
# ======================================================================
# [1] 공통 코드: 모델·학습 루프·평가 지표 (training/stage1.py train과 동일 절차)
#     + 태그 입력(features="tags")·태그 엣지(tag_edge_k) 옵션. 기본값이면 기존과 같은 결과
# ======================================================================
from __future__ import annotations

import copy
import csv
import json
import math
import os
import random
import statistics
from dataclasses import asdict, dataclass, replace
from pathlib import Path

# CUDA에서 결정론적 행렬곱(cuBLAS)에 필요: CUDA 초기화 전에 설정해야 함
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import torch
from torch import nn
from torch.nn import functional as F

# GPU가 있으면 GPU, 없으면 CPU로 학습
# (CPU는 단일 스레드로 training/stage1.py와 비트 단위까지 같은 결과, GPU는 연산 순서가 달라 수치가 조금 다름)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA = Path("data/stage1_rebuild_raw_gte.pt")
RESULTS = Path("results/experiments")
# 입력 특징 pt: raw = 기존 GTE 원본 입력, tags = 태그를 텍스트에 넣어 다시 인코딩한 입력
TAG_DATA = Path("tmp/stage1_raw_gte_tags.pt")
FEATURE_PATHS = {"raw": DATA, "tags": TAG_DATA}
PAIRS_CSV = Path("data/labeled_pairs_masked_v3.csv")  # 레시피별 태그(a_tags, b_tags) 출처
THRESHOLDS = (0.2, 0.4, 0.6, 0.8)  # 0~1 유사도에서 0/1/2/3/4점을 나누는 경계
AGGREGATORS = ("weighted_mean", "mean", "max", "gru")
GRU_ORDERS = ("fixed", "shuffle", "weight")


# ---------------- 데이터·손실·지표 (training/stage1.py와 동일) ----------------


def read_artifact(path: Path) -> dict:
    data = torch.load(path, map_location="cpu", weights_only=True)
    ids = data["recipe_ids"]
    features = data["recipe_features"]
    splits = data["splits"]
    if features.ndim != 2 or len(ids) != len(features):
        raise ValueError("unexpected recipe features")
    if not all(splits[part] for part in ("train", "val", "test")):
        raise ValueError("train, validation and test splits must be non-empty")
    if not torch.isfinite(features).all():
        raise ValueError("non-finite recipe features")
    return data


def context_edges(
    rows: list[tuple[int, int, int]],
) -> tuple[torch.Tensor, torch.Tensor]:
    positive = [(a, b, score / 4) for a, b, score in rows if score >= 1]
    if not positive:
        return torch.empty((2, 0), dtype=torch.long), torch.empty(0)
    forward = torch.tensor([(a, b) for a, b, _ in positive], dtype=torch.long).t()
    weights = torch.tensor([weight for _, _, weight in positive], dtype=torch.float32)
    return torch.cat((forward, forward.flip(0)), dim=1), weights.repeat(2)


def pair_tensors(
    rows: list[tuple[int, int, int]],
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    pairs = torch.tensor([(a, b) for a, b, _ in rows], dtype=torch.long).T.contiguous()
    classes = torch.tensor([score for _, _, score in rows], dtype=torch.long)
    labels = classes.float() / 4
    return pairs, labels, classes


def ordinal_loss(
    predictions: torch.Tensor,
    classes: torch.Tensor,
    temperature: float,
    last_boundary_weight: float = 1.0,
) -> torch.Tensor:
    thresholds = predictions.new_tensor(THRESHOLDS)
    targets = (classes[:, None] >= torch.arange(1, 5, device=classes.device)).float()
    logits = (predictions[:, None] - thresholds[None, :]) / temperature
    positive_weights = predictions.new_tensor((1, 1, 1, last_boundary_weight))
    return F.binary_cross_entropy_with_logits(
        logits, targets, pos_weight=positive_weights
    )


def measure(predictions: torch.Tensor, classes: torch.Tensor) -> dict:
    # 0~4점 변환: floor(5 * 유사도), positive 기준: 3점 이상
    predicted = (predictions * 5).floor().long().clamp(0, 4)
    labels = classes.float() / 4
    matrix = [
        [int(((classes == actual) & (predicted == guess)).sum()) for guess in range(5)]
        for actual in range(5)
    ]
    positive_actual, positive_predicted = classes >= 3, predicted >= 3
    tp = int((positive_actual & positive_predicted).sum())
    fp = int((~positive_actual & positive_predicted).sum())
    fn = int((positive_actual & ~positive_predicted).sum())
    tn = int((~positive_actual & ~positive_predicted).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    specificity = tn / (tn + fp) if tn + fp else 0.0
    return {
        "n": len(classes),
        "rmse_0_1": (predictions - labels).square().mean().sqrt().item(),
        "mae_0_4": (predictions - labels).abs().mean().item() * 4,
        "exact_accuracy": (predicted == classes).float().mean().item(),
        "within_one_accuracy": ((predicted - classes).abs() <= 1).float().mean().item(),
        "positive_precision": precision,
        "positive_recall": recall,
        "positive_f1": (
            2 * precision * recall / (precision + recall) if precision + recall else 0.0
        ),
        "balanced_accuracy": (recall + specificity) / 2,
        "confusion_matrix": matrix,
        "predicted_counts": torch.bincount(predicted, minlength=5).tolist(),
        "actual_counts": torch.bincount(classes, minlength=5).tolist(),
        "per_class_recall": [
            matrix[k][k] / sum(matrix[k]) if sum(matrix[k]) else 0.0 for k in range(5)
        ],
        "min_prediction": predictions.min().item(),
        "max_prediction": predictions.max().item(),
    }


# ---------------- 태그: 재료·영양·공통 분류 태그를 뺀 "조리법·완성 형태" 태그 ----------------

# 모든 레시피에 붙는 상위 분류 (정보 없음)
GENERIC_TAGS = set("""
preparation time-to-make course main-ingredient dietary occasion cuisine equipment
number-of-servings taste-mood low-in-something high-in-something free-of-something
""".split()) | {""}
# 재료를 드러내는 태그: "재료는 달라도 조리법·완성 형태가 비슷한 음식"이 목표이므로 제외
INGREDIENT_TAGS = set("""
meat vegetables fruit pasta-rice-and-grains eggs-dairy poultry chicken beef seafood pasta cheese
pork chocolate beans ground-beef fish potatoes eggs chicken-breasts rice berries nuts
saltwater-fish shellfish tropical-fruit grains apples tomatoes onions squash salmon citrus
shrimp mushrooms greens bananas coconut strawberries peppers broccoli spaghetti elbow-macaroni
pineapple pitted-fruit corn crab asparagus carrots ham bacon cauliflower lentils
long-grain-rice lemon freshwater-fish spinach chicken-thighs-legs pork-sausage blueberries
penne black-beans white-rice pork-loins turkey yams-sweet-potatoes tuna oranges
chick-peas-garbanzos lamb-sheep soy-tofu green-yellow-beans roast-beef raspberries peaches
steaks scallops wings pork-ribs mango lettuces cherries ravioli-tortellini lime pears gelatin
whole-chicken pasta-shells wild-game brown-rice steak beef-ribs bass deer veal catfish
tilapia beef-sausage clams plums melons grapes sole-and-flounder halibut cod turkey-breasts
mussels avocado a1-sauce peanut-butter oysters oatmeal mahi-mahi zucchini chicken-livers
trout papaya beef-organ-meats duck orange-roughy rabbit crawfish medium-grain-rice pheasant
short-grain-rice pickeral beef-liver kiwifruit whitefish duck-breasts bok-choys eggplant
""".split())
# 영양·식단 태그: 대부분 재료 구성으로 정해지므로 제외
DIET_TAGS = set("""
low-sodium healthy low-carb low-calorie low-cholesterol low-protein low-saturated-fat low-fat
healthy-2 vegan vegetarian very-low-carbs high-protein diabetic gluten-free egg-free
high-calcium lactose kosher dairy-free high-fiber nut-free non-alcoholic
""".split())
EXCLUDED_TAGS = GENERIC_TAGS | INGREDIENT_TAGS | DIET_TAGS


def recipe_tags(path: Path = PAIRS_CSV) -> dict[str, list[str]]:
    """레시피 ID -> 남길 태그 목록 (원래 순서 유지)"""
    import ast

    csv.field_size_limit(10**9)
    tags: dict[str, list[str]] = {}
    with Path(path).open(encoding="utf-8-sig", newline="") as handle:
        for row in csv.DictReader(handle):
            for side in ("a", "b"):
                rid = row[f"{side}_id"].strip()
                if rid not in tags:
                    tags[rid] = [
                        t for t in ast.literal_eval(row[f"{side}_tags"]) if t not in EXCLUDED_TAGS
                    ]
    return tags


def tag_knn_edges(
    recipe_ids: list, k: int, scale: float, path: Path = PAIRS_CSV
) -> tuple[torch.Tensor, torch.Tensor]:
    """태그 IDF 코사인 상위 k개 이웃을 양방향 연결 (가중치 = scale * 코사인). 라벨을 쓰지 않음"""
    tags = recipe_tags(path)
    ids = [str(rid) for rid in recipe_ids]
    sets = [set(tags.get(rid, [])) for rid in ids]
    vocab = sorted(set().union(*sets))
    column = {tag: index for index, tag in enumerate(vocab)}
    matrix = torch.zeros(len(ids), len(vocab))
    for row, tag_set in enumerate(sets):
        for tag in tag_set:
            matrix[row, column[tag]] = 1.0
    idf = torch.log(len(ids) / matrix.sum(0).clamp_min(1))
    vectors = F.normalize(matrix * idf, dim=1)
    similarity = vectors @ vectors.T
    similarity.fill_diagonal_(0)
    best: dict[tuple[int, int], float] = {}
    for node in range(len(ids)):
        order = torch.argsort(-similarity[node], stable=True)[:k]
        for other in order.tolist():
            value = float(similarity[node, other])
            if value <= 0:
                continue
            for edge in ((other, node), (node, other)):  # (source, target) 양방향
                best[edge] = max(best.get(edge, 0.0), value)
    edges = sorted(best)
    if not edges:
        return torch.empty((2, 0), dtype=torch.long), torch.empty(0)
    return (
        torch.tensor(edges, dtype=torch.long).T.contiguous(),
        torch.tensor([scale * best[e] for e in edges], dtype=torch.float32),
    )


# ---------------- 모델 ----------------


class NeighborAggregator(nn.Module):
    # 노드별 이웃 벡터를 하나로 모음 [노드 수, hidden] (이웃이 없는 노드는 0 벡터)
    def __init__(self, kind: str, hidden: int, gru_order: str = "fixed"):
        super().__init__()
        if kind not in AGGREGATORS:
            raise ValueError(f"unknown aggregator: {kind}")
        if gru_order not in GRU_ORDERS:
            raise ValueError(f"unknown gru_order: {gru_order}")
        self.kind = kind
        self.gru_order = gru_order
        if kind == "max":
            self.pool = nn.Linear(hidden, hidden)
        elif kind == "gru":
            self.gru = nn.GRU(hidden + 1, hidden, batch_first=True)

    def forward(
        self,
        x: torch.Tensor,
        edges: torch.Tensor,
        weights: torch.Tensor,
        generator: torch.Generator | None = None,
    ) -> torch.Tensor:
        if self.kind == "weighted_mean":
            # training/stage1.py WeightedSAGELayer와 같은 연산 순서
            neighbors = torch.zeros_like(x)
            counts = x.new_zeros(len(x))
            if weights.numel():
                source, target = edges
                neighbors.index_add_(0, target, x[source] * weights[:, None])
                counts.index_add_(0, target, weights)
            return neighbors / counts.clamp_min(1e-8)[:, None]
        if not edges.numel():
            return torch.zeros_like(x)
        source, target = edges
        if self.kind == "mean":
            neighbors = torch.zeros_like(x).index_add(0, target, x[source])
            counts = x.new_zeros(len(x)).index_add(0, target, torch.ones_like(weights))
            return neighbors / counts.clamp_min(1)[:, None]
        if self.kind == "max":
            messages = F.relu(self.pool(x[source])) * weights[:, None]
            # 메시지가 모두 0 이상이므로 0에서 시작해도 이웃 최댓값과 같음
            return torch.zeros_like(x).scatter_reduce(
                0, target[:, None].expand_as(messages), messages, reduce="amax"
            )
        return self._gru(x, source, target, weights, generator)

    def _neighbor_order(
        self,
        target: torch.Tensor,
        weights: torch.Tensor,
        generator: torch.Generator | None,
    ) -> torch.Tensor:
        # 엣지를 대상 노드별로 모으고, 노드 안에서는 gru_order 순서로 놓는 순열
        if self.gru_order == "shuffle" and self.training:
            # 셔플 난수 생성기는 CPU에 두고 순열만 학습 장치로 옮김
            first = torch.randperm(len(target), generator=generator).to(target.device)
        elif self.gru_order == "weight":
            first = torch.argsort(weights, stable=True)
        else:  # fixed (shuffle 모델의 평가 포함): 엣지 목록 순서
            first = torch.arange(len(target), device=target.device)
        return first[torch.argsort(target[first], stable=True)]

    def _gru(self, x, source, target, weights, generator) -> torch.Tensor:
        order = self._neighbor_order(target, weights, generator)
        source, target, weights = source[order], target[order], weights[order]
        nodes, counts = torch.unique_consecutive(target, return_counts=True)
        starts = torch.cumsum(counts, 0) - counts
        rows = torch.repeat_interleave(torch.arange(len(nodes), device=x.device), counts)
        positions = torch.arange(len(target), device=x.device) - torch.repeat_interleave(
            starts, counts
        )
        sequences = x.new_zeros(len(nodes), int(counts.max()), x.shape[1] + 1)
        sequences[rows, positions] = torch.cat((x[source], weights[:, None]), dim=1)
        packed = nn.utils.rnn.pack_padded_sequence(
            sequences, counts.cpu(), batch_first=True, enforce_sorted=False
        )
        _, state = self.gru(packed)
        return torch.zeros_like(x).index_copy(0, nodes, state[-1])


class SAGELayer(nn.Module):
    # GraphSAGE 계층: 자기 벡터 변환 + 집계한 이웃 벡터 변환
    def __init__(self, hidden: int, aggregator: str, gru_order: str):
        super().__init__()
        self.self_linear = nn.Linear(hidden, hidden)
        self.neighbor_linear = nn.Linear(hidden, hidden, bias=False)
        self.aggregate = NeighborAggregator(aggregator, hidden, gru_order)

    def forward(self, x, edges, weights, generator=None) -> torch.Tensor:
        # 이웃 집계를 먼저 계산해야 기존 모델과 역전파 합산 순서가 같아 결과가 동일함
        neighbors = self.aggregate(x, edges, weights, generator)
        return self.self_linear(x) + self.neighbor_linear(neighbors)


class ExperimentPairGraphSAGE(nn.Module):
    """RecipePairGraphSAGE에서 집계 방식과 레이어 수만 바꿀 수 있게 한 실험용 모델

    레이어 구성은 기존과 같다: 입력 사영 -> [dropout -> SAGE -> LayerNorm -> GELU] x num_layers
    (2번째 레이어부터 residual 연결). num_layers=0이면 그래프를 쓰지 않는 기준 모델이다.
    """

    def __init__(
        self,
        input_dim: int,
        hidden: int,
        dropout: float,
        raw_cosine_stats: tuple[float, float] | None = None,
        num_layers: int = 2,
        aggregator: str = "weighted_mean",
        gru_order: str = "fixed",
    ):
        super().__init__()
        if num_layers < 0:
            raise ValueError("num_layers must be non-negative")
        self.dropout = dropout
        self.use_raw_cosine = raw_cosine_stats is not None
        if raw_cosine_stats is not None:
            self.register_buffer("raw_cosine_mean", torch.tensor(raw_cosine_stats[0]))
            self.register_buffer("raw_cosine_std", torch.tensor(raw_cosine_stats[1]))
        self.input = nn.Sequential(
            nn.Linear(input_dim, hidden), nn.LayerNorm(hidden), nn.GELU()
        )
        # 기존 모델과 같은 순서로 파라미터를 만들어 같은 시드면 같은 초기값이 되게 함
        self.layers = nn.ModuleList()
        self.norms = nn.ModuleList()
        for _ in range(num_layers):
            self.layers.append(SAGELayer(hidden, aggregator, gru_order))
            self.norms.append(nn.LayerNorm(hidden))
        self.pair_head = nn.Sequential(
            nn.Linear(hidden * 3 + int(self.use_raw_cosine), hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )

    def encode(
        self,
        features: torch.Tensor,
        edges: torch.Tensor,
        weights: torch.Tensor,
        generator: torch.Generator | None = None,
    ) -> torch.Tensor:
        x = self.input(features)
        for index, (layer, norm) in enumerate(zip(self.layers, self.norms)):
            x = F.dropout(x, p=self.dropout, training=self.training)
            hidden = layer(x, edges, weights, generator)
            x = F.gelu(norm(hidden + x if index else hidden))
        return x

    def score(
        self,
        embeddings: torch.Tensor,
        pairs: torch.Tensor,
        raw_features: torch.Tensor | None = None,
    ) -> torch.Tensor:
        left, right = embeddings[pairs[0]], embeddings[pairs[1]]
        symmetric = torch.cat(
            (torch.abs(left - right), left * right, (left + right) / 2), dim=-1
        )
        if self.use_raw_cosine:
            raw_cosine = F.cosine_similarity(
                raw_features[pairs[0]], raw_features[pairs[1]]
            )
            standardized = (raw_cosine - self.raw_cosine_mean) / self.raw_cosine_std
            symmetric = torch.cat((symmetric, standardized[:, None]), dim=-1)
        return torch.sigmoid(self.pair_head(symmetric).squeeze(-1))


# ---------------- 학습 (training/stage1.py train과 같은 절차) ----------------


@dataclass(frozen=True)
class Config:
    # 기본값 = 현재 1단계 최적 설정 (GTE 원본 입력, hidden 64)
    aggregator: str = "weighted_mean"
    gru_order: str = "fixed"
    num_layers: int = 2
    lr: float = 0.003
    hidden: int = 64
    dropout: float = 0.15
    weight_decay: float = 0.001
    ordinal_weight: float = 0.2
    temperature: float = 0.1
    folds: int = 5
    epochs: int = 100
    patience: int = 10
    raw_cosine_feature: bool = True
    features: str = "raw"  # FEATURE_PATHS의 키: raw(기존 입력) / tags(태그 포함 입력)
    tag_edge_k: int = 0  # 0이면 태그 엣지 없음 (기존과 동일)
    tag_edge_weight: float = 0.5  # 태그 엣지 가중치 = 이 값 * 태그 코사인
    seed: int = 42
    # 학습 장치도 설정에 포함: CPU/GPU 결과는 수치가 달라 이어 하기 캐시에서 섞이지 않게 함
    device: str = DEVICE


def train_once(config: Config, data_path: Path = DATA, verbose: bool = False) -> dict:
    """설정 하나를 학습하고 metrics.json과 같은 형식의 결과(+history)를 반환"""
    if config.ordinal_weight < 0 or config.temperature <= 0 or config.folds < 2:
        raise ValueError("invalid training settings")
    device = torch.device(config.device)
    # 결정론적 실행: 같은 데이터·시드·장치면 항상 같은 결과
    if device.type == "cpu":
        torch.set_num_threads(1)
        torch.use_deterministic_algorithms(True)
    else:
        # 결정론적 구현이 없는 CUDA 연산은 오류 대신 경고 (해당 연산만 미세하게 달라질 수 있음)
        torch.use_deterministic_algorithms(True, warn_only=True)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    torch.manual_seed(config.seed)
    random.seed(config.seed)
    data_path = FEATURE_PATHS.get(config.features, data_path)
    artifact = read_artifact(Path(data_path))
    features = artifact["recipe_features"]
    splits = artifact["splits"]
    # 태그 엣지: 라벨과 무관하므로 학습 폴드·검증·테스트 모든 그래프에 항상 포함
    tag_edges = None
    if config.tag_edge_k:
        tag_edges = tag_knn_edges(artifact["recipe_ids"], config.tag_edge_k, config.tag_edge_weight)

    def graph(rows):
        edges, weights = context_edges(rows)
        if tag_edges is None:
            return edges, weights  # 기존 모델과 완전히 같은 그래프
        return torch.cat((edges, tag_edges[0]), dim=1), torch.cat((weights, tag_edges[1]))
    raw_cosine_stats = None
    if config.raw_cosine_feature:
        training_pairs, _, _ = pair_tensors(splits["train"])
        training_cosine = F.cosine_similarity(
            features[training_pairs[0]], features[training_pairs[1]]
        )
        raw_cosine_stats = (
            training_cosine.mean().item(),
            training_cosine.std(unbiased=False).clamp_min(1e-6).item(),
        )
    model = ExperimentPairGraphSAGE(
        features.shape[1],
        config.hidden,
        config.dropout,
        raw_cosine_stats,
        config.num_layers,
        config.aggregator,
        config.gru_order,
    ).to(device)  # CPU에서 시드로 초기화한 뒤 옮기므로 초기 가중치는 장치와 무관하게 같음
    features = features.to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=config.lr, weight_decay=config.weight_decay
    )
    # 이웃 셔플 전용 난수 (dropout 난수열과 분리)
    generator = torch.Generator().manual_seed(config.seed)
    val_pairs, val_labels, val_classes = (
        t.to(device) for t in pair_tensors(splits["val"])
    )
    val_edges, val_weights = (t.to(device) for t in graph(splits["train"]))
    rng = random.Random(config.seed)
    best_objective, best_epoch, stale = math.inf, 0, 0
    best_state = None
    history = []
    epoch = 0
    # 폴드 교대 제외로 모든 학습 쌍을 예측 대상에 포함, 현재 폴드의 SIMILAR 연결은 입력 그래프에서 제외
    for epoch in range(1, config.epochs + 1):
        order = list(range(len(splits["train"])))
        rng.shuffle(order)
        fold_indices = [order[i :: config.folds] for i in range(config.folds)]
        train_sse = 0.0
        train_n = 0
        model.train()
        for selected in fold_indices:
            excluded = set(selected)
            context = [
                row
                for index, row in enumerate(splits["train"])
                if index not in excluded
            ]
            targets = [splits["train"][index] for index in selected]
            edges, weights = (t.to(device) for t in graph(context))
            pairs, labels, classes = (t.to(device) for t in pair_tensors(targets))
            optimizer.zero_grad()
            predictions = model.score(
                model.encode(features, edges, weights, generator), pairs, features
            )
            rmse = ((predictions - labels).square().mean() + 1e-8).sqrt()
            loss = rmse + config.ordinal_weight * ordinal_loss(
                predictions, classes, config.temperature
            )
            loss.backward()
            optimizer.step()
            train_sse += float((predictions.detach() - labels).square().sum())
            train_n += len(labels)
        model.eval()
        with torch.no_grad():
            val_predictions = model.score(
                model.encode(features, val_edges, val_weights), val_pairs, features
            )
            val_rmse = (val_predictions - val_labels).square().mean().sqrt().item()
            val_ord = ordinal_loss(
                val_predictions, val_classes, config.temperature
            ).item()
            val_objective = val_rmse + config.ordinal_weight * val_ord
            val_exact = (
                ((val_predictions * 5).floor().long().clamp(0, 4) == val_classes)
                .float()
                .mean()
                .item()
            )
        if not math.isfinite(val_objective):
            raise ValueError("non-finite validation objective")
        train_rmse = (train_sse / train_n) ** 0.5
        history.append(
            {
                "epoch": epoch,
                "train_rmse_0_1": train_rmse,
                "val_rmse_0_1": val_rmse,
                "val_ordinal_loss": val_ord,
                "val_objective": val_objective,
                "val_exact_accuracy": val_exact,
            }
        )
        if val_objective < best_objective:
            best_objective, best_epoch, stale = val_objective, epoch, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            stale += 1
        if verbose and (epoch == 1 or epoch % 10 == 0):
            print(
                f"epoch={epoch} train_rmse={train_rmse:.4f} "
                f"val_rmse={val_rmse:.4f} val_exact={val_exact:.1%}",
                flush=True,
            )
        if stale >= config.patience:
            break
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        embeddings = model.encode(features, val_edges, val_weights)
        val_predictions = model.score(embeddings, val_pairs, features)
        test_pairs, _, test_classes = (t.to(device) for t in pair_tensors(splits["test"]))
        test_predictions = model.score(embeddings, test_pairs, features)
    return {
        "settings": asdict(config),
        "best_epoch": best_epoch,
        "stopped_epoch": epoch,
        # 조기 종료 없이 epoch 상한까지 간 경우: 학습률이 낮아 덜 수렴했을 수 있음
        "hit_epoch_cap": stale < config.patience,
        "parameters": sum(p.numel() for p in model.parameters()),
        "val": measure(val_predictions, val_classes),
        "test": measure(test_predictions, test_classes),
        "history": history,
    }


# ---------------- 실험 반복 (조건 x 시드) 및 요약 ----------------

SCALAR_METRICS = (
    "rmse_0_1",
    "mae_0_4",
    "exact_accuracy",
    "within_one_accuracy",
    "positive_precision",
    "positive_recall",
    "positive_f1",
    "balanced_accuracy",
)


def _flatten(label: str, report: dict) -> dict:
    row = {
        "condition": label,
        "seed": report["settings"]["seed"],
        "best_epoch": report["best_epoch"],
        "stopped_epoch": report["stopped_epoch"],
        "hit_epoch_cap": report["hit_epoch_cap"],
        "parameters": report["parameters"],
    }
    for part in ("val", "test"):
        for name in SCALAR_METRICS:
            row[f"{part}_{name}"] = report[part][name]
        # 4점 적중 수 (실제 4점을 4점으로 예측한 개수)
        row[f"{part}_hits_4"] = report[part]["confusion_matrix"][4][4]
    return row


def run_grid(
    name: str,
    conditions: dict[str, dict],
    seeds: list[int],
    base: Config = Config(),
    data_path: Path = DATA,
    root: Path = RESULTS,
) -> list[dict]:
    """조건(이름 -> Config에서 바꿀 값) x 시드를 모두 학습하고 결과 행 목록을 반환

    결과는 root/name/<조건>/seed<시드>/metrics.json에 저장하며, 같은 설정으로 이미 학습한
    결과가 있으면 다시 학습하지 않고 읽는다 (중간에 멈춰도 이어서 실행 가능).
    """
    rows = []
    total = len(conditions) * len(seeds)
    for label, overrides in conditions.items():
        for seed in seeds:
            config = replace(base, **overrides, seed=seed)
            out = Path(root) / name / label / f"seed{seed}"
            path = out / "metrics.json"
            report = None
            if path.exists():
                saved = json.loads(path.read_text(encoding="utf-8"))
                if saved["settings"] == asdict(config):
                    report = saved
            if report is None:
                report = train_once(config, data_path)
                history = report.pop("history")
                out.mkdir(parents=True, exist_ok=True)
                path.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
                (out / "history.json").write_text(
                    json.dumps(history, indent=2) + "\n", encoding="utf-8"
                )
            rows.append(_flatten(label, report))
            print(
                f"[{len(rows)}/{total}] {label} seed={seed} "
                f"best_epoch={report['best_epoch']} "
                f"val_rmse={report['val']['rmse_0_1']:.4f} "
                f"val_exact={report['val']['exact_accuracy']:.1%}",
                flush=True,
            )
    _write_csv(Path(root) / name / "runs.csv", rows)
    summary = summarize(rows, conditions, base)
    _write_csv(
        Path(root) / name / "summary.csv",
        [{k: v for k, v in row.items() if k != "config"} for row in summary],
    )
    (Path(root) / name / "summary.json").write_text(
        json.dumps(summary, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
    )
    return rows


def summarize(
    rows: list[dict], conditions: dict[str, dict], base: Config = Config()
) -> list[dict]:
    # 조건별 시드 평균·표준편차 (검증 RMSE 평균 오름차순)
    summary = []
    for label, overrides in conditions.items():
        group = [row for row in rows if row["condition"] == label]
        if not group:
            continue
        entry = {
            "condition": label,
            "runs": len(group),
            # 시드·장치는 다음 실험에 넘기지 않음 (다음 실험은 현재 세션의 장치 사용)
            "config": {
                k: v
                for k, v in asdict(replace(base, **overrides)).items()
                if k not in ("seed", "device")
            },
            "device": base.device,
            "epoch_cap_runs": sum(row["hit_epoch_cap"] for row in group),
            "parameters": group[0]["parameters"],
        }
        for key in ["best_epoch"] + [
            f"{part}_{name}"
            for part in ("val", "test")
            for name in (*SCALAR_METRICS, "hits_4")
        ]:
            values = [float(row[key]) for row in group]
            entry[f"{key}_mean"] = statistics.fmean(values)
            entry[f"{key}_std"] = statistics.stdev(values) if len(values) > 1 else 0.0
        summary.append(entry)
    return sorted(summary, key=lambda entry: entry["val_rmse_0_1_mean"])


def best_config(name: str, root: Path = RESULTS) -> tuple[str, dict]:
    """이전 실험 요약에서 검증 RMSE 평균이 가장 낮은 조건의 (이름, 설정) 반환"""
    summary = json.loads((Path(root) / name / "summary.json").read_text(encoding="utf-8"))
    return summary[0]["condition"], summary[0]["config"]


def markdown_table(summary: list[dict], noise: float | None = None) -> str:
    """요약 표 (평균 ± 표준편차). 1위와의 검증 RMSE 차이가 noise보다 작은 조건에 ≈ 표시 (시드 노이즈 수준)"""
    columns = (
        ("val_rmse_0_1", "검증 RMSE", 4),
        ("val_exact_accuracy", "검증 정확", 3),
        ("val_within_one_accuracy", "검증 ±1", 3),
        ("val_positive_f1", "검증 F1(≥3)", 3),
        ("val_hits_4", "검증 4점 적중", 1),
        ("test_rmse_0_1", "테스트 RMSE", 4),
        ("test_exact_accuracy", "테스트 정확", 3),
        ("best_epoch", "best epoch", 1),
    )
    header = "| 순위 | 조건 | " + " | ".join(title for _, title, _ in columns) + " |"
    lines = [header, "|" + "---|" * (len(columns) + 2)]
    best = summary[0]["val_rmse_0_1_mean"] if summary else 0.0
    if noise is None and summary:
        noise = summary[0]["val_rmse_0_1_std"]  # 기본: 1위 조건의 시드 간 표준편차
    for rank, entry in enumerate(summary, 1):
        cells = [
            f"{entry[f'{key}_mean']:.{digits}f} ± {entry[f'{key}_std']:.{digits}f}"
            for key, _, digits in columns
        ]
        mark = ""
        if noise is not None and rank > 1 and entry["val_rmse_0_1_mean"] - best < noise:
            mark = " ≈"
        if entry["epoch_cap_runs"]:
            mark += f" (상한 도달 {entry['epoch_cap_runs']}회)"
        lines.append(f"| {rank} | {entry['condition']}{mark} | " + " | ".join(cells) + " |")
    return "\n".join(lines)


def _write_csv(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def neighborhood_reach(data_path: Path = DATA, max_hops: int = 3) -> list[dict]:
    """학습 그래프(학습 쌍 중 1점 이상)에서 k홉 안에 닿는 이웃 수 통계 (레이어 수 해석용)"""
    artifact = read_artifact(Path(data_path))
    count = len(artifact["recipe_ids"])
    edges, _ = context_edges(artifact["splits"]["train"])
    adjacency = torch.zeros(count, count, dtype=torch.bool)
    adjacency[edges[0], edges[1]] = True
    reach = torch.eye(count, dtype=torch.bool)
    previous = reach.sum(1)
    connected = adjacency.any(1)
    rows = []
    for hops in range(1, max_hops + 1):
        reach = reach | (reach.float() @ adjacency.float()).bool()
        sizes = reach.sum(1) - 1  # 자기 자신 제외
        rows.append(
            {
                "hops": hops,
                "nodes_with_neighbors": int((sizes > 0).sum()),
                "mean_reach_connected": sizes[connected].float().mean().item(),
                "max_reach": int(sizes.max()),
                # 이 홉에서 새 이웃이 생긴 노드 수 (레이어를 하나 더 쌓아 새 정보가 들어오는 노드)
                "nodes_gaining": int((reach.sum(1) > previous).sum()) if hops > 1 else int(connected.sum()),
            }
        )
        previous = reach.sum(1)
    return rows


def plot_summary(summary: list[dict], title: str) -> None:
    # 조건별 검증/테스트 RMSE 평균 ± 표준편차 (Colab 기본 폰트에 한글이 없어 그래프 글자는 영어)
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print("matplotlib이 없어 그래프를 건너뜀")
        return
    labels = [entry["condition"] for entry in summary]
    positions = range(len(labels))
    fig, ax = plt.subplots(figsize=(max(6, 1.2 * len(labels)), 4))
    for offset, part in ((-0.18, "val"), (0.18, "test")):
        ax.errorbar(
            [x + offset for x in positions],
            [entry[f"{part}_rmse_0_1_mean"] for entry in summary],
            yerr=[entry[f"{part}_rmse_0_1_std"] for entry in summary],
            fmt="o", capsize=4, label=part,
        )
    ax.set_xticks(list(positions), labels, rotation=20)
    ax.set_ylabel("RMSE (0-1, lower is better)")
    ax.set_title(title)
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_histories(name: str, labels: list[str], seed: int, root: Path = RESULTS) -> None:
    # 조건별 epoch에 따른 검증 RMSE (시드 하나)
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print("matplotlib이 없어 그래프를 건너뜀")
        return
    fig, ax = plt.subplots(figsize=(8, 4))
    for label in labels:
        path = Path(root) / name / label / f"seed{seed}" / "history.json"
        if not path.exists():
            continue
        history = json.loads(path.read_text(encoding="utf-8"))
        ax.plot([h["epoch"] for h in history], [h["val_rmse_0_1"] for h in history], label=label)
    ax.set_xlabel("epoch")
    ax.set_ylabel("val RMSE")
    ax.set_title(f"{name}: learning curves (seed {seed})")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

In [2]:
# ======================================================================
# [2] 공통 설정
# ======================================================================
from IPython.display import Markdown, display

SEEDS = [42, 43, 44, 45, 46]
RECIPES_CSV = Path("data/recipes_raw_v3.csv")  # 레시피 이름·조리 단계 (태그 입력 인코딩용)

for path in (DATA, RECIPES_CSV, PAIRS_CSV):
    if not path.exists():
        raise FileNotFoundError(f"{path}가 없습니다. 저장소 루트에서 실행하세요.")
print("torch", torch.__version__, "| 장치", DEVICE, "| 시드", SEEDS)

tags = recipe_tags()
artifact = read_artifact(DATA)
ids = [str(r) for r in artifact["recipe_ids"]]
from collections import Counter
counter = Counter(t for rid in ids for t in tags.get(rid, []))
print(f"레시피 {len(ids)}개 | 남긴 태그 종류 {len(counter)}개 | 레시피당 평균 {sum(counter.values()) / len(ids):.1f}개 "
      f"| 태그 없는 레시피 {sum(1 for rid in ids if not tags.get(rid))}개")
print("자주 쓰인 태그:", ", ".join(f"{t}({n})" for t, n in counter.most_common(25)))


torch 2.15.0+cpu | 장치 cpu | 시드 [42, 43, 44, 45, 46]
레시피 2914개 | 남긴 태그 종류 266개 | 레시피당 평균 7.1개 | 태그 없는 레시피 1개
자주 쓰인 태그: easy(1491), main-dish(932), 60-minutes-or-less(924), 30-minutes-or-less(679), 4-hours-or-less(629), north-american(612), desserts(557), 15-minutes-or-less(510), 3-steps-or-less(463), holiday-event(441), dinner-party(439), oven(424), inexpensive(403), beginner-cook(395), american(382), 5-ingredients-or-less(353), side-dishes(350), kid-friendly(339), comfort-food(322), european(297), weeknight(292), stove-top(271), presentation(269), lunch(268), for-1-or-2(265)


## 태그 입력 준비

기존 1단계 텍스트에 태그를 이름 바로 뒤에 넣어 다시 인코딩합니다(512토큰을 넘는 긴 레시피도 태그가 첫 창에 들어가도록). 먼저 태그 없는 원래 텍스트 일부를 다시 인코딩해 저장된 임베딩과 같은지 확인합니다.

In [3]:
# ======================================================================
# 태그 입력 pt 만들기 (이미 있으면 건너뜀)
# ======================================================================
from src.engine import GTETextEncoder
from training.stage1 import recipe_text

csv.field_size_limit(10**9)
with RECIPES_CSV.open(encoding="utf-8-sig", newline="") as handle:
    recipe_rows = {}
    for row in csv.DictReader(handle):
        recipe_rows.setdefault(row["id"].strip(), row)


def tagged_text(row: dict) -> str:
    base = recipe_text(row)  # "Recipe: {이름}. Steps: {...}"
    tag_list = [t.replace("-", " ") for t in tags.get(row["id"].strip(), [])]
    if not tag_list:
        return base
    head = f"Recipe: {row['name']}. "
    return head + f"Tags: {', '.join(tag_list)}. " + base[len(head):]


print("예시:", tagged_text(recipe_rows[ids[0]])[:300], "...")

if TAG_DATA.exists():
    print(f"{TAG_DATA} 있음: 다시 인코딩하지 않음")
else:
    encoder = GTETextEncoder()
    check = list(range(0, len(ids), len(ids) // 40))[:40]
    again = encoder.encode_texts([recipe_text(recipe_rows[ids[i]]) for i in check]).cpu()
    agreement = (again * artifact["recipe_features"][check]).sum(1)
    print(f"태그 없는 텍스트 재인코딩 vs 저장된 임베딩 코사인: min={agreement.min():.6f}")
    if agreement.min() < 0.999:
        raise ValueError("기존 임베딩을 재현하지 못함: 인코더·텍스트 설정을 확인하세요")
    features = encoder.encode_texts(
        [tagged_text(recipe_rows[rid]) for rid in ids], show_progress_bar=True
    ).cpu()
    TAG_DATA.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {**{k: artifact[k] for k in ("recipe_ids", "splits", "seed")},
         "recipe_features": features, "encoder": encoder.MODEL_NAME, "view": "raw+tags",
         "encoding_strategy": "ordered_token_windows_v1"},
        TAG_DATA,
    )
    print(f"saved {TAG_DATA}")

tag_artifact = read_artifact(TAG_DATA)
shift = (tag_artifact["recipe_features"] * artifact["recipe_features"]).sum(1)
print(f"태그 입력 vs 기존 입력 임베딩 코사인: 평균 {shift.mean():.4f}, 최소 {shift.min():.4f}")


예시: Recipe: arriba baked winter squash mexican style. Tags: 60 minutes or less, north american, side dishes, mexican, easy, fall, holiday event, winter, christmas, seasonal. Steps: make a choice and proceed with recipe depending on size of squash , cut into half or fourths remove seeds for spicy squash  ...
태그 없는 텍스트 재인코딩 vs 저장된 임베딩 코사인: min=0.999999
saved tmp\stage1_raw_gte_tags.pt
태그 입력 vs 기존 입력 임베딩 코사인: 평균 0.9805, 최소 0.9341


## 태그 엣지가 그래프를 얼마나 채우는지

학습 그래프(라벨 엣지)만 쓸 때와 태그 엣지를 더할 때, 검증·테스트 쌍의 레시피 중 이웃이 있는 비율입니다.

In [4]:
# ======================================================================
# 그래프 커버리지
# ======================================================================
def coverage(edges: torch.Tensor, rows) -> tuple[float, float]:
    has = torch.zeros(len(ids), dtype=torch.bool)
    if edges.numel():
        has[edges[1]] = True
    ends = [(has[a].item(), has[b].item()) for a, b, _ in rows]
    return (sum(x + y for x, y in ends) / (2 * len(ends)), sum(not (x or y) for x, y in ends) / len(ends))


label_edges, _ = context_edges(artifact["splits"]["train"])
lines = ["| 그래프 | 엣지 수(방향 포함) | 검증 레시피 중 이웃 있음 | 양쪽 다 이웃 없는 검증 쌍 | 이웃 있는 노드의 평균 이웃 수 |",
         "|---|---|---|---|---|"]
for name, k in (("라벨 엣지만", 0), ("+ 태그 k=5", 5), ("+ 태그 k=10", 10)):
    edges = label_edges
    if k:
        edges = torch.cat((label_edges, tag_knn_edges(ids, k, 0.5)[0]), dim=1)
    with_nbr, none = coverage(edges, artifact["splits"]["val"])
    degree = torch.bincount(edges[1], minlength=len(ids)).float()
    lines.append(f"| {name} | {edges.shape[1]} | {with_nbr:.1%} | {none:.1%} | {degree[degree > 0].mean():.2f} |")
display(Markdown("\n".join(lines)))


| 그래프 | 엣지 수(방향 포함) | 검증 레시피 중 이웃 있음 | 양쪽 다 이웃 없는 검증 쌍 | 이웃 있는 노드의 평균 이웃 수 |
|---|---|---|---|---|
| 라벨 엣지만 | 3812 | 61.2% | 36.4% | 2.03 |
| + 태그 k=5 | 25284 | 100.0% | 0.0% | 8.68 |
| + 태그 k=10 | 44634 | 100.0% | 0.0% | 15.32 |

## 실험: 태그 입력 · 태그 엣지

조건 선택은 원본과 같이 **검증 RMSE 평균**으로 하고, 테스트 지표는 참고용입니다. 1위와의 검증 RMSE 차이가 1위의 시드 표준편차보다 작으면 ≈를 붙입니다.

In [5]:
# ======================================================================
# 실험 4. 태그 입력 · 태그 엣지
# ======================================================================
EXPERIMENT_4 = "04_tags"
BASE_4 = Config()  # 현재 1단계 최적 설정 (weighted_mean, 레이어 2, lr 0.003, hidden 64)
CONDITIONS_4 = {
    "baseline": {},
    "tag_input": {"features": "tags"},
    "tag_edges_k5": {"tag_edge_k": 5},
    "tag_edges_k10": {"tag_edge_k": 10},
    "tag_input+tag_edges_k5": {"features": "tags", "tag_edge_k": 5},
}

rows_4 = run_grid(EXPERIMENT_4, CONDITIONS_4, SEEDS, BASE_4, DATA, RESULTS)
summary_4 = summarize(rows_4, CONDITIONS_4, BASE_4)
display(Markdown(markdown_table(summary_4)))
plot_summary(summary_4, "Exp 4. tags as input / as graph edges")


[1/25] baseline seed=42 best_epoch=61 val_rmse=0.1810 val_exact=52.7%
[2/25] baseline seed=43 best_epoch=63 val_rmse=0.1825 val_exact=57.1%
[3/25] baseline seed=44 best_epoch=48 val_rmse=0.1883 val_exact=54.6%
[4/25] baseline seed=45 best_epoch=48 val_rmse=0.1860 val_exact=56.1%
[5/25] baseline seed=46 best_epoch=61 val_rmse=0.1849 val_exact=56.3%
[6/25] tag_input seed=42 best_epoch=54 val_rmse=0.1885 val_exact=52.1%
[7/25] tag_input seed=43 best_epoch=53 val_rmse=0.1954 val_exact=54.2%
[8/25] tag_input seed=44 best_epoch=54 val_rmse=0.1937 val_exact=54.8%
[9/25] tag_input seed=45 best_epoch=59 val_rmse=0.1954 val_exact=52.1%
[10/25] tag_input seed=46 best_epoch=62 val_rmse=0.1956 val_exact=52.9%
[11/25] tag_edges_k5 seed=42 best_epoch=77 val_rmse=0.1805 val_exact=54.8%
[12/25] tag_edges_k5 seed=43 best_epoch=85 val_rmse=0.1885 val_exact=53.1%
[13/25] tag_edges_k5 seed=44 best_epoch=70 val_rmse=0.1881 val_exact=56.5%
[14/25] tag_edges_k5 seed=45 best_epoch=89 val_rmse=0.1926 val_exact=

| 순위 | 조건 | 검증 RMSE | 검증 정확 | 검증 ±1 | 검증 F1(≥3) | 검증 4점 적중 | 테스트 RMSE | 테스트 정확 | best epoch |
|---|---|---|---|---|---|---|---|---|---|
| 1 | baseline | 0.1845 ± 0.0029 | 0.554 ± 0.017 | 0.936 ± 0.005 | 0.793 ± 0.013 | 27.2 ± 2.6 | 0.1890 ± 0.0020 | 0.517 ± 0.008 | 56.2 ± 7.5 |
| 2 | tag_edges_k5 | 0.1876 ± 0.0044 | 0.542 ± 0.020 | 0.926 ± 0.010 | 0.799 ± 0.006 | 32.2 ± 6.8 | 0.2042 ± 0.0032 | 0.503 ± 0.011 | 80.8 ± 7.4 |
| 3 | tag_input | 0.1937 ± 0.0030 | 0.532 ± 0.012 | 0.921 ± 0.006 | 0.775 ± 0.003 | 28.4 ± 5.6 | 0.1980 ± 0.0047 | 0.498 ± 0.011 | 56.4 ± 3.9 |
| 4 | tag_edges_k10 | 0.1964 ± 0.0091 | 0.510 ± 0.019 | 0.922 ± 0.007 | 0.768 ± 0.028 | 28.6 ± 6.6 | 0.2102 ± 0.0059 | 0.488 ± 0.016 | 75.2 ± 9.4 |
| 5 | tag_input+tag_edges_k5 | 0.2061 ± 0.0100 | 0.484 ± 0.026 | 0.912 ± 0.013 | 0.744 ± 0.032 | 26.8 ± 5.9 | 0.2076 ± 0.0092 | 0.474 ± 0.015 | 57.6 ± 11.9 |

matplotlib이 없어 그래프를 건너뜀


In [6]:
# ======================================================================
# 조건별 클래스 재현율 (검증, 시드 평균) · 정확도 차이
# ======================================================================
by = {}
for label in CONDITIONS_4:
    recalls, exact = [], []
    for seed in SEEDS:
        report = json.loads((RESULTS / EXPERIMENT_4 / label / f"seed{seed}" / "metrics.json").read_text(encoding="utf-8"))
        recalls.append(report["val"]["per_class_recall"])
        exact.append(report["val"]["exact_accuracy"])
    by[label] = (torch.tensor(recalls).mean(0).tolist(), statistics.fmean(exact))
lines = ["| 조건 | 0점 | 1점 | 2점 | 3점 | 4점 | 검증 정확 (baseline 대비) |", "|---|---|---|---|---|---|---|"]
for label, (recall, exact) in by.items():
    diff = exact - by["baseline"][1]
    lines.append(f"| {label} | " + " | ".join(f"{r:.2f}" for r in recall) + f" | {exact:.3f} ({diff:+.3f}) |")
display(Markdown("\n".join(lines)))


| 조건 | 0점 | 1점 | 2점 | 3점 | 4점 | 검증 정확 (baseline 대비) |
|---|---|---|---|---|---|---|
| baseline | 0.77 | 0.33 | 0.28 | 0.76 | 0.44 | 0.554 (+0.000) |
| tag_input | 0.76 | 0.21 | 0.26 | 0.75 | 0.46 | 0.532 (-0.021) |
| tag_edges_k5 | 0.73 | 0.33 | 0.35 | 0.67 | 0.52 | 0.542 (-0.012) |
| tag_edges_k10 | 0.67 | 0.27 | 0.39 | 0.62 | 0.46 | 0.510 (-0.043) |
| tag_input+tag_edges_k5 | 0.64 | 0.25 | 0.39 | 0.57 | 0.43 | 0.484 (-0.070) |

## 결과 다운로드

`results/experiments/` 전체를 zip으로 묶어 내려받습니다(Colab 전용).

In [ ]:
import shutil

archive = shutil.make_archive("stage1_tag_experiments", "zip", RESULTS)
try:
    from google.colab import files

    files.download(archive)
except ImportError:
    print(f"Colab이 아니므로 다운로드 대신 파일만 만듦: {archive}")